In [25]:
!pip install --upgrade unsloth unsloth_zoo
# !pip install vllm

In [26]:
!pip install -U torchvision

In [1]:
import os, math, numpy as np
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
# os.environ["VLLM_USE_V1"] = "0"

In [2]:
from huggingface_hub import login
from google.colab import userdata
HF_key = userdata.get('PLo_HF')
login(token = HF_key)
model_save = 'awilliam60412/0827-Qwen2.5-32B-16bit-1E'

In [3]:
import pandas as pd
from google.colab import drive
import torch
drive.mount('/content/drive', force_remount=True)
FOLDERNAME = 'Colab Notebooks'
%cd drive/MyDrive/$FOLDERNAME/
test = pd.read_csv(f"Peter/data/test_data.csv")
sub = test[['row_id']].copy()
# examples = pd.read_csv(f'Peter/data/few_shot_examples.csv')

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks


In [4]:
import os
import math
import numpy as np
import pandas as pd
import torch
from unsloth import FastLanguageModel
from transformers import LogitsProcessor, LogitsProcessorList
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
# --- 1. THE FIX: A TRANSFORMERS-COMPATIBLE LOGITS PROCESSOR ---
# Since Unsloth uses the standard Hugging Face `generate`, we use this style of processor.
class YesNoLogitsProcessor(LogitsProcessor):
    # ✅ FIX 1: The __init__ method must accept the 'model' to get its vocab size.
    def __init__(self, model, tokenizer):
        self.tokenizer = tokenizer
        self.no_token_id = tokenizer.encode("No", add_special_tokens=False)[0]
        self.yes_token_id = tokenizer.encode("Yes", add_special_tokens=False)[0]

        # ✅ FIX 2: Create the mask using model.config.vocab_size. This guarantees the shape will match.
        self.allowed_mask = torch.zeros(model.config.vocab_size, dtype=torch.bool)

        self.allowed_mask[self.no_token_id] = True
        self.allowed_mask[self.yes_token_id] = True
        print(f"Constraining to tokens: No ({self.no_token_id}), Yes ({self.yes_token_id})")
        print(f"Mask shape created with model's vocab size: {self.allowed_mask.shape[0]}")

    # ✅ FIX 3: You must include the __call__ method for the class to work.
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        """This method is called by the generator to process logits."""
        mask = self.allowed_mask.to(scores.device)

        # ✅ THE FIX: Apply the 1D mask to the first row of the 2D scores tensor.
        scores[0, ~mask] = -float('inf')

        return scores

In [6]:
# --- 2. MODEL LOADING with UNSLOTH ---
max_seq_length = 3072
dtype = None # None for auto detection
load_in_4bit = False

print("Loading model with Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_save,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Model and tokenizer loaded.")
FastLanguageModel.for_inference(model)

Loading model with Unsloth...


AttributeError: module 'transformers.models.bit.modeling_bit' has no attribute 'Linear'

In [ ]:
def formatting(dataset):
    texts = []
    for i in range(len(dataset)):
        texts.append(tokenizer.apply_chat_template(dataset[i], tokenize=False, add_generation_prompt=False))
    return texts

In [ ]:
sys_prompt = ""
dataset = []
all_prompts = []

In [ ]:
# --- 3. PROMPT & DATA SETUP (Same as before) ---
# (The template and prompt formatting logic remains the same as your original code)
template = """Subreddit: r/{subreddit}
Rule: {rule}
Examples:
1) {positive_example_1}
Violation: Yes
2) {negative_example_1}
Violation: No
3) {negative_example_2}
Violation: No
4) {positive_example_2}
Violation: Yes
Comment:
{body}
Violation:"""

sys_prompt_template = """
Rule: {rule}
Example: {example}
Violation: {Label}
"""
# Reasoning: {reasoning}

sys_prompt = 'You are given a comment on reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No. No need to explain.'

# sys_prompt += "Here are some examples for your reference."
# for index,row in examples.iterrows():
#   if row.Label == 1:
#     llabel = "Yes"
#   else:
#     llabel = "No"
#   print(llabel)
#   few_shot = sys_prompt_template.format(rule = row.rule, example = row.example, Label = llabel)
#   # few_shot = sys_prompt_template.format(rule = row.rule, example = row.example, Label = row.Label, reasoning = row.reasoning)
#   sys_prompt += few_shot


dataset = []
for index,row in test.iterrows():

    formatted_sample = [
        {
        "role": "system",
        "content": sys_prompt
    },
       {
           "role": "user",
           "content": template.format(
               rule = row.rule,
               subreddit = row.subreddit,
               body = row.body,
               positive_example_1 = row.positive_example_1,
               negative_example_1 = row.negative_example_1,
               positive_example_2 = row.positive_example_2,
               negative_example_2 = row.negative_example_2
           )
       }]

    dataset.append( formatted_sample )
all_prompts = formatting(dataset)
# dataset = []
# for index, row in test.iterrows():
#     dataset.append([
#         {"role": "system", "content": sys_prompt},
#         {"role": "user", "content": template.format(**row.to_dict())},
#     ])
# all_prompts = [tokenizer.apply_chat_template(d, tokenize=False, add_generation_prompt=True) for d in dataset]

In [ ]:
# --- 4. GENERATION AND PROBABILITY EXTRACTION ---
# Instantiate the logits processor
# ✅ FIX 4: Pass BOTH the model and tokenizer to the processor.
yes_no_processor = YesNoLogitsProcessor(model, tokenizer)
logits_processor_list = LogitsProcessorList([yes_no_processor])

print("Generating responses and calculating probabilities...")
# The generation part remains the same
responses = []
for prompt in tqdm(all_prompts):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=2,
        logits_processor=logits_processor_list,
        output_scores=True,
        return_dict_in_generate=True
    )
    responses.append(outputs)

In [ ]:
# --- 5. PROCESS RESULTS & CALCULATE AUC ---
probs_for_yes = []
predictions = [] # New list to store 0s and 1s

print("\n--- Processing Results ---")
for i, response in enumerate(responses):
    # Get the generated text
    generated_text = tokenizer.decode(response.sequences[0, -1]).strip()

    # ✅ **THE CHANGE IS HERE**
    # Convert text to integer prediction
    if generated_text == "Yes":
        prediction = 1
    else:
        prediction = 0
    predictions.append(prediction)

    # Probability calculation remains the same for AUC score
    first_token_logits = response.scores[0][0]
    yes_logit = first_token_logits[yes_no_processor.yes_token_id].item()
    no_logit = first_token_logits[yes_no_processor.no_token_id].item()
    softmax_probs = torch.nn.functional.softmax(torch.tensor([no_logit, yes_logit]), dim=0)
    normalized_yes_prob = softmax_probs[1].item()
    probs_for_yes.append(normalized_yes_prob)

    # Updated print statement
    # print(f"Comment {i+1}: Generated Text: '{generated_text}' -> Prediction: {prediction}")

In [ ]:
print(probs_for_yes)

In [ ]:
ans = pd.read_csv(f"Peter/data/Ans.csv")
count = 0
for i in range(len(ans)):
  if predictions[i] >=0.5:
    val_ans = 1
  else:
    val_ans = 0
  if ans["rule_violation"][i] == val_ans:
    count += 1
print(count/len(ans))

In [ ]:
predictions

In [ ]:
sub['rule_violation'] = predictions

In [ ]:
sub.to_csv('submission-4B-base')

In [ ]:
prob = test[['row_id']].copy()

In [ ]:
prob['rule_violation'] = probs_for_yes

In [ ]:
prob.to_csv('prob_submission-4B-base.csv')